# 00 — One-Time Data Preprocessing


In [14]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT_ROOT = Path("/content/drive/MyDrive/jakirJisan")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_ROOT}. "
        "Make sure the extracted folder is directly inside MyDrive."
    )

print("Project root:", PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/jakirJisan


In [15]:
import shutil
import gdown
from pathlib import Path

# Exact Google Drive target path
raw_dir = Path(PROJECT_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

# 1. Move the files that were downloaded to /data/raw
if Path("/data/raw").exists():
    for f in Path("/data/raw").glob("*.csv"):
        shutil.copy(f, raw_dir / f.name)
        print(f"Copied from /data/raw: {f.name} -> {raw_dir}")

# 2. Download any remaining files directly to Google Drive
files = {
    "BanglaRQA_targeted_domains.csv": "18w1wty89i9osl51ad1iQAHkxobP89x0B",
    "validation_targeted.csv": "1JTkFEIcwXkutDjhXdbWfHazv7Zz4FZdE",
    "test_targeted.csv": "1XkdTDdw4_IshTLMnQuprFJcJVhPTftaX",
}

for fname, file_id in files.items():
    dest = raw_dir / fname
    if not dest.exists():
        print(f"Downloading {fname} directly to {dest}...")
        gdown.download(id=file_id, output=str(dest), quiet=False)
    else:
        print(f"Verified present: {dest.name} ({dest.stat().st_size} bytes)")

print("\nFiles now confirmed in your Google Drive:")
for p in raw_dir.glob("*.csv"):
    print(" -", p)


Copied from /data/raw: test_targeted.csv -> /content/drive/MyDrive/jakirJisan/data/raw
Copied from /data/raw: validation_targeted.csv -> /content/drive/MyDrive/jakirJisan/data/raw
Copied from /data/raw: BanglaRQA_targeted_domains.csv -> /content/drive/MyDrive/jakirJisan/data/raw
Verified present: BanglaRQA_targeted_domains.csv (17087637 bytes)
Verified present: validation_targeted.csv (4225662 bytes)
Verified present: test_targeted.csv (4620052 bytes)

Files now confirmed in your Google Drive:
 - /content/drive/MyDrive/jakirJisan/data/raw/test_targeted.csv
 - /content/drive/MyDrive/jakirJisan/data/raw/validation_targeted.csv
 - /content/drive/MyDrive/jakirJisan/data/raw/BanglaRQA_targeted_domains.csv


In [16]:
import subprocess
import sys

requirements_file = PROJECT_ROOT / "requirements" / "preprocessing.txt"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)
])
print("Preprocessing dependencies installed.")

Preprocessing dependencies installed.


In [17]:
import sys

project_path = str(PROJECT_ROOT)
if project_path not in sys.path:
    sys.path.insert(0, project_path)

In [18]:
from src.config import ProjectConfig
from src.controllers import PreprocessingController

project = ProjectConfig(project_root=PROJECT_ROOT)
processed, tokenized = PreprocessingController(project).run()

Tokenizing train:   0%|          | 0/3035 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/831 [00:00<?, ? examples/s]

Tokenizing test:   0%|          | 0/886 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3035 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/831 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/886 [00:00<?, ? examples/s]

PREPROCESSING COMPLETE
TRAIN      | rows=3035 | unanswerable=725 | missing_targets= 0
VALIDATION | rows= 831 | unanswerable=204 | missing_targets= 0
TEST       | rows= 886 | unanswerable=229 | missing_targets= 0
Max source length: 768
Max target length: 96
Max tokenized train source: 768
Max tokenized train target: 96


In [19]:
prepared = PROJECT_ROOT / "data" / "prepared"
required = [
    "train_processed.parquet",
    "validation_processed.parquet",
    "test_processed.parquet",
    "tokenized_train",
    "tokenized_validation",
    "tokenized_test",
    "tokenizer",
    "metadata.json",
]
missing = [name for name in required if not (prepared / name).exists()]
print("Prepared data folder:", prepared)
print("Missing artifacts:", missing)
assert not missing, f"Preprocessing incomplete: {missing}"
print("Preprocessing artifacts verified successfully.")

Prepared data folder: /content/drive/MyDrive/jakirJisan/data/prepared
Missing artifacts: []
Preprocessing artifacts verified successfully.
